<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day15_practice2_%EC%9E%84%EB%B2%A0%EB%94%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 임베딩 - 의미를 담는 숫자

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [3]:
# 셀 1. 미니 감성 코퍼스
data = [ # (문장, 긍정1/부정0)
    ("이 영화 정말 재미있다", 1), ("정말 최고의 영화", 1),
    ("연기가 훌륭하다", 1), ("스토리가 재미있다", 1),
    ("배우가 최고다", 1), ("정말 훌륭하다", 1),
    ("음악이 아름답다", 1), ("연출이 훌륭하다", 1),
    ("최고다 재미있다", 1), ("아름답다 최고다", 1),
    ("이 영화 너무 지루하다", 0), ("최악이다 정말 지루하다", 0),
    ("연기가 어색하다", 0), ("스토리가 지루하다", 0),
    ("배우가 최악이다", 0), ("정말 어색하다", 0),
    ("음악이 끔찍하다", 0), ("연출이 최악이다", 0),
    ("끔찍하다 지루하다", 0), ("어색하다 끔찍하다", 0),
]

In [5]:
# 단어 사전 만들기
vocab = {"<pad>": 0, "<unk>": 1}
for s, _ in data:
  for tok in s.split():
    vocab.setdefault(tok, len(vocab)) # 없을 때만 추가
print(f"사전 크기: {len(vocab)}")
print(vocab)

사전 크기: 20
{'<pad>': 0, '<unk>': 1, '이': 2, '영화': 3, '정말': 4, '재미있다': 5, '최고의': 6, '연기가': 7, '훌륭하다': 8, '스토리가': 9, '배우가': 10, '최고다': 11, '음악이': 12, '아름답다': 13, '연출이': 14, '너무': 15, '지루하다': 16, '최악이다': 17, '어색하다': 18, '끔찍하다': 19}


In [6]:
# 2. 정수 인코딩 - 각 문장을 정수 텐서로
def encode(s, max_len=4):
  ids = [vocab.get(t, 1) for t in s.split()][:max_len] # 문장을 번호 리스트로 변환 후 앞 4개만
  return ids + [0] * (max_len - len(ids))
X = torch.tensor([encode(s) for s, _ in data])
y = torch.tensor([lab for _, lab in data], dtype=torch.float32).reshape(-1, 1) # 2차원 배열
print(X)
print(y)

tensor([[ 2,  3,  4,  5],
        [ 4,  6,  3,  0],
        [ 7,  8,  0,  0],
        [ 9,  5,  0,  0],
        [10, 11,  0,  0],
        [ 4,  8,  0,  0],
        [12, 13,  0,  0],
        [14,  8,  0,  0],
        [11,  5,  0,  0],
        [13, 11,  0,  0],
        [ 2,  3, 15, 16],
        [17,  4, 16,  0],
        [ 7, 18,  0,  0],
        [ 9, 16,  0,  0],
        [10, 17,  0,  0],
        [ 4, 18,  0,  0],
        [12, 19,  0,  0],
        [14, 17,  0,  0],
        [19, 16,  0,  0],
        [18, 19,  0,  0]])
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])


In [8]:
# 셀 2. 임베딩 (nn.Embedding)

EMB_DIM = 8 # 단어 하나 = 숫자 8개
emb = nn.Embedding(len(vocab), EMB_DIM, padding_idx=0) # 단어사전 크기, 벡터 차원 수, 0벡터(학습 제외)
print(emb)
print(emb.weight)
# 임베딩: 단어 하나를 벡터로 만든다. 단어 하나(번호)를 정해진 길이의 숫자 목록으로 바꾼다. 처음엔 이 8개 숫자가 그냥 랜덤이라 아무 의미가 없다
# 학습하면서 이 숫자들도 함께 갱신되고, 학습이 끝나면 비슷한 뜻의 단어는 벡터가 서로 가까워지고, 반대 뜻은 멀어진다
# 이 상대적 위치 관계가 곧 의미이다. 임베딩 학습은 이 "의미의 지도"에서 각 단어의 자리를 잡아주는 과정이고 비슷한 단어끼리 가까이 모이도록 배치한다

Embedding(20, 8, padding_idx=0)
Parameter containing:
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-1.6022,  1.3529,  1.2888,  0.0523, -1.5469,  0.7567,  0.7755,  2.0265],
        [ 0.0358,  0.1206, -0.8057, -0.2076, -0.9319, -1.5910, -1.1360, -0.5226],
        [-0.5188, -1.5013, -1.9267,  0.1279,  1.0229, -0.5558,  0.7043,  0.7099],
        [ 1.7744, -0.9216,  0.9624, -0.3370, -1.1753,  0.3581,  0.4788,  1.3537],
        [ 0.5261,  2.1120, -0.5208, -0.9320,  0.1852,  1.0687,  1.3065,  0.4598],
        [-0.8146, -1.0212, -0.4949, -0.5923,  0.1543,  0.4408, -0.1483, -2.3184],
        [-0.3980,  1.0805, -1.7809,  1.5080,  0.3094, -0.5003,  1.0350,  1.6896],
        [-0.0045,  1.6668,  0.1539, -1.0603, -0.5727,  0.0836,  0.3999,  1.9892],
        [-0.0720, -0.9061, -2.0487, -1.0811,  0.0176,  0.0782,  0.1932,  0.4097],
        [-0.9291,  0.2762, -0.5389,  0.4626, -0.8719, -0.0271, -0.3532,  1.4639],
        [ 1.2554, -0.7150,  0.8539,  0.5130,

In [12]:
idx = torch.tensor(vocab["영화"]) # 단어사전 3번
print("numbedding(영화) :", [round(v, 2) for v in emb(idx).tolist()])
print("weight[영화 행] :", [round(v, 2) for v in emb.weight[idx].tolist()])
assert torch.equal(emb(idx), emb.weight[idx])
print("완전 일치: 임베딩 = 표에서 해당 행을 꺼내는 것")
# 임베딩 = (사전크기20 × 8차원벡터) 표 하나
# 최신 LLM 임베딩 = 단어사전 20만개 × 1만 차원의 벡터

numbedding(영화) : [-0.52, -1.5, -1.93, 0.13, 1.02, -0.56, 0.7, 0.71]
weight[영화 행] : [-0.52, -1.5, -1.93, 0.13, 1.02, -0.56, 0.7, 0.71]
완전 일치: 임베딩 = 표에서 해당 행을 꺼내는 것


In [13]:
# 셀 3. 학습 전 스냅샷 - 지금 유사도는 '무의미한 랜덤'
def sim(w1, w2, table):
  v1, v2 = table[vocab[w1]], table[vocab[w2]] # 두 단어의 벡터를 임베딩(표)에서 꺼냄
  return F.cosine_similarity(v1, v2, dim=0).item() # 코사인 유사도: 두 벡터의 '방향'이 비슷한가

before = emb.weight.detach().clone()
pairs = [("재미있다", "최고다"), ("재미있다", "지루하다"), ("훌륭하다", "최악이다")]
print("[학습 전 - 랜덤 초기값]")
for a, b in pairs:
  print(f"{a} ↔ {b}: {sim(a, b, before):+.2f}")

[학습 전 - 랜덤 초기값]
재미있다 ↔ 최고다: -0.05
재미있다 ↔ 지루하다: -0.40
훌륭하다 ↔ 최악이다: +0.22


In [14]:
# 셀 4. 감성 분류로 임베딩 학습 - 의미가 스며든다
class TinySentiment(nn.Module):
  def __init__(self):
    super().__init__()
    self.emb = emb # 위에서 만든 임베딩(표) 그대로 사용
    self.fc = nn.Linear(EMB_DIM, 1) # 문장 벡터(8차원) → 점수 1개 (긍정/부정 판단부)
  def forward(self, x): # x: (B, 4) 각 문장이 단어 4개
    vecs = self.emb(x) # 1. 문장이 들어오면 각 단어를 벡터로 바꾸고
    sent = vecs.mean(dim=1) # 2. 그 벡터들을 평균 내서 문장 하나를 대표하는 벡터로 만들고
    return torch.sigmoid(self.fc(sent)) # 3. 그걸로 긍정 확률(0~1)을 뱉는다

model = TinySentiment()
loss_fn = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(300):
  loss = loss_fn(model(X), y)
  opt.zero_grad()
  loss.backward()
  opt.step()
print(f"학습 완료 (loss {loss.item():.4f})")

after = emb.weight.detach() # 학습 후 임베딩
print("\n[학습 후 - 감성 과제를 풀며 생긴 의미]")
for a, b in pairs:
  print(f"{a} ↔ {b}: 학습 전 {sim(a, b, before):+.2f} → 학습 후 {sim(a, b, after):+.2f}")

학습 완료 (loss 0.0003)

[학습 후 - 감성 과제를 풀며 생긴 의미]
재미있다 ↔ 최고다: 학습 전 -0.05 → 학습 후 +0.77
재미있다 ↔ 지루하다: 학습 전 -0.40 → 학습 후 -0.90
훌륭하다 ↔ 최악이다: 학습 전 +0.22 → 학습 후 -0.80


In [16]:
print(before[3]) # 학습 전 임베딩 3번행
print(after[3]) # 학습 후 임베딩 3번행 (의미가 생김)

tensor([-0.5188, -1.5013, -1.9267,  0.1279,  1.0229, -0.5558,  0.7043,  0.7099])
tensor([-2.0865, -0.0795, -0.3862,  1.9898, -0.7089, -2.2112,  0.1574,  0.2725])
